# Census Data Ingestion - Bronze Layer

Consolidated ingestion of Census boundaries and demographics into Unity Catalog.

**Data Sources:**
- Census Cartographic Boundary Files via `pygris` (500k resolution)
- ACS 5-Year demographic data via Census API

**Output Tables:**
- `{catalog}.{bronze_schema}.census_blockgroups` - Block group boundaries
- `{catalog}.{bronze_schema}.census_states` - State boundaries
- `{catalog}.{bronze_schema}.census_demographics` - ACS demographic data

In [ ]:
# MAGIC %md
# MAGIC ## Parameters

In [ ]:
import pygris
from pygris import states, block_groups
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid
import geopandas as gpd
import requests
import yaml

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("state_fips", "25")  # Default: Massachusetts
dbutils.widgets.text("year", "2023")
dbutils.widgets.text("census_api_key", "")
dbutils.widgets.text("config_path", "")

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
state_fips = dbutils.widgets.get("state_fips")
year = int(dbutils.widgets.get("year")) if dbutils.widgets.get("year") else 2023
census_api_key = dbutils.widgets.get("census_api_key")
config_path = dbutils.widgets.get("config_path")

assert catalog and bronze_schema, "Missing required parameters: catalog, bronze_schema"

# Output table names
bg_table = f"{catalog}.{bronze_schema}.census_blockgroups"
states_table = f"{catalog}.{bronze_schema}.census_states"
demographics_table = f"{catalog}.{bronze_schema}.census_demographics"

print(f"Catalog: {catalog}")
print(f"Schema: {bronze_schema}")
print(f"State FIPS: {state_fips}")
print(f"Year: {year}")

In [ ]:
# MAGIC %md
# MAGIC ## Helper Functions

In [ ]:
def geopandas_to_spark_with_geometry(gdf, geography_level, ingest_id, ingest_timestamp):
    """
    Convert GeoPandas GeoDataFrame to Spark DataFrame with native GEOGRAPHY type.
    Uses WKT format for efficient conversion to Databricks ST functions.
    
    Args:
        gdf: GeoPandas GeoDataFrame from pygris
        geography_level: 'block_group' or 'state'
        ingest_id: UUID for tracking ingestion batch
        ingest_timestamp: Timestamp of ingestion
    
    Returns:
        Spark DataFrame with native GEOGRAPHY column (SRID 4326)
    """
    gdf_copy = gdf.copy()
    gdf_copy['geometry_wkt'] = gdf_copy['geometry'].apply(lambda geom: geom.wkt if geom is not None else None)
    gdf_copy = gdf_copy.drop(columns=['geometry'])
    
    spark_df = spark.createDataFrame(gdf_copy)
    
    # Convert WKT to native GEOGRAPHY type with SRID 4326 (WGS 84)
    spark_df = spark_df.withColumn(
        "geometry",
        F.expr("ST_GeomFromText(geometry_wkt, 4326)")
    ).drop("geometry_wkt")
    
    # Add ingestion metadata
    spark_df = (spark_df
                .withColumn("geography_level", F.lit(geography_level))
                .withColumn("ingestion_id", F.lit(ingest_id))
                .withColumn("ingestion_timestamp", F.lit(ingest_timestamp)))
    
    return spark_df


def get_census_data(geography_level, state_fips, variables_dict, api_key, year):
    """
    Fetch ACS 5-Year data from Census API.
    Block groups require full geographic hierarchy.
    """
    base_url = f"https://api.census.gov/data/{year}/acs/acs5"
    var_string = ",".join(variables_dict.keys())
    
    url = f"{base_url}?get=NAME,{var_string}&for=block%20group:*&in=state:{state_fips}&in=county:*&in=tract:*&key={api_key}"
    
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    
    data = response.json()
    assert data and len(data) >= 2, f"Invalid API response for {geography_level}"
    
    return (data[0], data[1:])


def transform_demographics_to_dataframe(headers, rows, geography_level, variables_dict, ingest_id, ingest_timestamp, acs_year):
    """Transform Census API response to Spark DataFrame with type casting and metadata."""
    df = spark.createDataFrame(rows, schema=headers)
    
    # Rename to friendly names
    for census_code, friendly_name in variables_dict.items():
        if census_code in df.columns:
            df = df.withColumnRenamed(census_code, friendly_name)
    
    # Add metadata
    df = (df
          .withColumn("geography_level", F.lit(geography_level))
          .withColumn("acs_year", F.lit(acs_year))
          .withColumn("ingestion_id", F.lit(ingest_id))
          .withColumn("ingestion_timestamp", F.lit(ingest_timestamp)))
    
    # Cast numeric columns
    geo_cols = ["NAME", "state", "county", "tract", "block_group", 
                "geography_level", "acs_year", "ingestion_id", "ingestion_timestamp"]
    for col_name in df.columns:
        if col_name not in geo_cols:
            df = df.withColumn(col_name, F.col(col_name).cast("long"))
    
    return df

In [ ]:
# MAGIC %md
# MAGIC ## Section 1: Census Boundaries (Block Groups & States)

In [ ]:
# Generate ingestion metadata
ingest_id = str(uuid.uuid4())
ingest_timestamp = datetime.now()

print(f"Ingestion ID: {ingest_id}")
print(f"Fetching block groups for state FIPS {state_fips}, year {year}...")

# Fetch Block Groups for specified state using pygris
bg_gdf = block_groups(
    state=state_fips,
    county=None,
    year=year,
    cache=True,
    cb=True  # Cartographic boundaries (500k resolution)
)
print(f"Fetched {len(bg_gdf)} block groups")

# Fetch all US states
print("Fetching state boundaries...")
states_gdf = states(
    cb=True,
    resolution='500k',
    year=year
)
print(f"Fetched {len(states_gdf)} states")

In [ ]:
# Convert to Spark DataFrames with geometry
bg_df = geopandas_to_spark_with_geometry(bg_gdf, "block_group", ingest_id, ingest_timestamp)
state_df = geopandas_to_spark_with_geometry(states_gdf, "state", ingest_id, ingest_timestamp)

# Standardize block group columns
bg_df = (bg_df
         .withColumnRenamed("GEOID", "geoid")
         .withColumnRenamed("NAME", "name")
         .withColumnRenamed("STATEFP", "state_fips")
         .withColumnRenamed("COUNTYFP", "county_fips")
         .withColumnRenamed("TRACTCE", "tract")
         .withColumnRenamed("BLKGRPCE", "block_group_id")
         .withColumnRenamed("ALAND", "area_land")
         .withColumnRenamed("AWATER", "area_water"))

# Standardize state columns
state_df = (state_df
            .withColumnRenamed("GEOID", "geoid")
            .withColumnRenamed("STUSPS", "state_abbr")
            .withColumnRenamed("NAME", "name")
            .withColumnRenamed("STATEFP", "state_fips")
            .withColumnRenamed("ALAND", "area_land")
            .withColumnRenamed("AWATER", "area_water"))

print(f"Block groups schema: {bg_df.columns}")
print(f"States schema: {state_df.columns}")

In [ ]:
# Write block groups to Unity Catalog
(bg_df
 .repartition(10)
 .write
 .mode("overwrite")
 .option("mergeSchema", "true")
 .option("overwriteSchema", "true")
 .saveAsTable(bg_table))
print(f"Written {bg_df.count()} block groups to {bg_table}")

# Write states to Unity Catalog
(state_df
 .repartition(1)
 .write
 .mode("overwrite")
 .option("mergeSchema", "true")
 .option("overwriteSchema", "true")
 .saveAsTable(states_table))
print(f"Written {state_df.count()} states to {states_table}")

In [ ]:
# MAGIC %md
# MAGIC ## Section 2: Census Demographics (ACS 5-Year)

In [ ]:
# Skip demographics if API key not provided
if not census_api_key or not config_path:
    print("Skipping demographics ingestion: census_api_key or config_path not provided")
    dbutils.notebook.exit("Boundaries ingested successfully. Demographics skipped (no API key).")

In [ ]:
# Load census variables from YAML config
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Flatten nested structure
census_variables = {}
for category, variables in config['acs_5_year_variables'].items():
    census_variables.update(variables)

print(f"Loaded {len(census_variables)} census variables from config")

In [ ]:
# Fetch demographics from Census API
print(f"Fetching ACS 5-Year demographics for state {state_fips}, year {year}...")
bg_headers, bg_rows = get_census_data("block_group", state_fips, census_variables, census_api_key, year)
print(f"Fetched {len(bg_rows)} block group records")

# Transform to Spark DataFrame
bg_headers = [h.replace("block group", "block_group") for h in bg_headers]
demographics_df = transform_demographics_to_dataframe(
    bg_headers, bg_rows, "block_group", census_variables, ingest_id, ingest_timestamp, str(year)
)

In [ ]:
# Write demographics to Unity Catalog
(demographics_df
 .write
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable(demographics_table))

print(f"Written {demographics_df.count()} demographic records to {demographics_table}")

In [ ]:
# MAGIC %md
# MAGIC ## Validation

In [ ]:
print("=" * 80)
print("CENSUS DATA INGESTION VALIDATION")
print("=" * 80)

# Validate block groups
print(f"\n1. Block Groups Table ({bg_table}):")
bg_validation = spark.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(geometry) as non_null_geometries,
        ST_SRID(FIRST(geometry)) as srid
    FROM {bg_table}
""")
bg_validation.show(truncate=False)

# Validate states
print(f"\n2. States Table ({states_table}):")
state_validation = spark.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(geometry) as non_null_geometries,
        ST_SRID(FIRST(geometry)) as srid
    FROM {states_table}
""")
state_validation.show(truncate=False)

# Validate demographics (if ingested)
try:
    print(f"\n3. Demographics Table ({demographics_table}):")
    demo_validation = spark.sql(f"""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT state) as states,
            COUNT(DISTINCT county) as counties
        FROM {demographics_table}
    """)
    demo_validation.show(truncate=False)
except Exception as e:
    print(f"Demographics table not available: {e}")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)